# pydbsp integration

*Warning:* This is a low-level chapter. You don't have to read it unless you are interested in the inner workings of Kafi Streams.


## Overview

[Preparation](#prep)

* [Input/output](#input_output)
  * [to_zSet()](#to_zSet)
    * [from_records](#from_records)
    * [from_debezium](#from_debezium)
    * [_from_records](#_from_records)
  * [from_zSet()](#from_zSet)
    * [to_records](#to_records)
    * [to_debezium](#to_debezium)
    * [_to_records](#_to_records)
* [Pack/unpack](#pack_unpack)
  * [pack_fun](#pack_fun)
  * [unpack_fun](#unpack_fun)
* [State](#state)
  * [get_state()](#get_state)
  * [set_state()](#set_state)
  * [load_state()](#load_state)
  * [save_state()](#save_state)
  * [get_state_size()](#get_state_size)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [1]:
import sys
sys.path.insert(1, ".")
sys.path.insert(1, "../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator
click_generator = ClickGenerator()
debezium_click_generator = ClickGenerator(debezium_bool=True)
weights_click_generator = ClickGenerator(weights_bool=True)

click_source_str = "clicks"
sink_str = "sink"


---
<a id="input_output"></a>
## Input/output

In broad strokes, a Kafi Streams processing step involves the following four sub steps:
1. New data comes in from append-only source streams.
2. The records are converted to *Z-sets* for pydbsp.
3. pydbsp processes the new data and returns the resulting changes, again as *Z-sets*. Each sink receives one.
4. These *Z-sets* are converted to append-only sink streams.
5. The resulting changes come out in the append-only sink streams.

Here is a graphical representation of a Kafi Streams processing step:
```mermaid
flowchart LR
  
  Input@{ shape: lean-r, label: "1. Input stream (append-only)" }

  Kafi(("3. Kafi Streams processing\n(pydbsp; Z-sets)"))

  Output@{ shape: lean-r, label: "5. Output stream (append-only)" }

  Input -.-> |2. convert\nto Z-Set| Kafi
  Kafi -.-> |4. convert\nfrom Z-set| Output
```

As you can see, the steps *2* and *4* involve conversions. You can configure how these conversions work using the operators:
* [to_zSet()](#to_zSet) - configure the conversion from input records to Z-sets
* [from_zSet()](#from_zSet) - configure the conversion from Z-sets to output records


<a id="to_zset"></a>
### to_zSet()

The `to_zSet()` operator allows you to specify what kind of input stream Kafi Streams expects for the source and how this input stream is converted into *Z-sets*.

Currently, Kafi Streams offers three options:
* `from_records` This is the default - expects a list of records and converts it into a Z-set where each record gets the weight *1*. 
* `from_debezium` Expects a list of records in *Debezium* format and converts it into the corresponding Z-set.
* `_from_records` Expects a list of pairs of a record and a weight and converts into the corresponding Z-set (only makes sense for *TopologyNode*, not for *Streams*)


<a id="from_records"></a>
#### from_records

This is the default option. Lists of records, typically Kafka messages, come in and each of them gets weight `1`.

Here is an example where you can see that each input record simply gets weight `1`:


In [2]:
sink_tn = Tn.source(click_source_str).to_zSet(Tn.from_records)._peek().sink(sink_str)
#
tn = Tn.build(sink_tn)
#
m_list = click_generator.generate(2)

print("Input records:")
for m in m_list:
    print(m)

print("\nZ-set:")
_ = tn.process({click_source_str: m_list})


Input records:
{'key': None, 'value': {'customer_id': 72, 'view_time': 104, 'ts': 1787588207715}}
{'key': None, 'value': {'customer_id': 73, 'view_time': 108, 'ts': 1787588207815}}

Z-set:
({'key': None, 'value': {'customer_id': 72, 'view_time': 104, 'ts': 1787588207715}}, 1)
({'key': None, 'value': {'customer_id': 73, 'view_time': 108, 'ts': 1787588207815}}, 1)


<a id="from_debezium"></a>
#### from_debezium

Append-only streams from Kafka can also include weights in the sense of DBSP, e.g. if using the *Debezium* format where each record is marked by an operation type in the `op` field of the `value` field.

Here is an example where we generate two input records using the Debezium format:
1. a *change* (`"op": "c"`) that gets weight `1` in pydbsp
2. a *delete* (`"op": "d"`) that gets weight `-1` in pydbsp


In [3]:
sink_tn = Tn.source(click_source_str).to_zSet(Tn.from_debezium)._peek().sink(sink_str)
#
tn = Tn.build(sink_tn)
#
m_list = debezium_click_generator.generate(1, w=1) + debezium_click_generator.generate(1, w=-1)

print("Input records:")
for m in m_list:
    print(m)

print("\nZ-set:")
_ = tn.process({click_source_str: m_list})


Input records:
{'key': None, 'value': {'customer_id': 8, 'view_time': 87, 'ts': 1787588207715, 'before': None, 'after': {'customer_id': 8, 'view_time': 87, 'ts': 1787588207715}, 'op': 'c'}}
{'key': None, 'value': {'customer_id': 22, 'view_time': 45, 'ts': 1787588207815, 'before': {'customer_id': 22, 'view_time': 45, 'ts': 1787588207815}, 'after': None, 'op': 'd'}}

Z-set:
({'key': None, 'value': {'customer_id': 8, 'view_time': 87, 'ts': 1787588207715}}, 1)
({'key': None, 'value': {'customer_id': 22, 'view_time': 45, 'ts': 1787588207815}}, -1)


<a id="_from_records"></a>
#### _from_records

This option allows you to explicitly specify the weights of the incoming records.

Here is an example:


In [4]:
sink_tn = Tn.source(click_source_str).to_zSet(Tn._from_records)._peek().sink(sink_str)
#
tn = Tn.build(sink_tn)
#
m_w_tuple_list = weights_click_generator.generate(1, w=1) + weights_click_generator.generate(1, w=-1)

print("Input record/weight tuples:")
for m_w_tuple in m_w_tuple_list:
    print(m_w_tuple)

print("\nZ-set:")
_ = tn.process({click_source_str: m_w_tuple_list})


Input record/weight tuples:
({'key': None, 'value': {'customer_id': 84, 'view_time': 13, 'ts': 1787588207715}}, 1)
({'key': None, 'value': {'customer_id': 1, 'view_time': 39, 'ts': 1787588207815}}, -1)

Z-set:
({'key': None, 'value': {'customer_id': 84, 'view_time': 13, 'ts': 1787588207715}}, 1)
({'key': None, 'value': {'customer_id': 1, 'view_time': 39, 'ts': 1787588207815}}, -1)


<a id="from_zset"></a>
### from_zSet()

The `from_zSet()` operator allows you to specify how to convert the resulting *Z-sets* into output records.

Currently, Kafi Streams offers three options:
* `to_records` This is the default - converts the resulting Z-sets into lists of records. Each record with a positive weight `w` is converted into `w` output records.
* `to_debezium` Conerts the resulting Z-sets into lists of records in *Debezium* format. Each record with a positive weight `w` is converted into `w` output records with Debezium `op` set to `c`. Each record with a negative weight `w` is converted into `w` output records with Debezium `op` set to `d`.
* `_to_records` Converts the resulting Z-sets into lists of pairs of a record and its weight (only makes sense for *TopologyNode*, not for *Streams*)



<a id="to_records"></a>
#### to_records

This is the default - converts the resulting Z-sets into lists of records. Each record with a positive weight `w` is converted into `w` output records.

Here is an example:


In [5]:
sink_tn = Tn.source(click_source_str).to_zSet(Tn.from_records)._peek().sink(sink_str)
#
tn = Tn.build(sink_tn).from_zSet(Tn.to_records)
#
m_list = click_generator.generate(2)

print("Input records:")
for m in m_list:
    print(m)

print("\npydbsp Z-set (as a list of pairs):")
m_list = tn.process({click_source_str: m_list})[sink_str]

print("\nOutput records:")
for m in m_list:
    print(m)


Input records:
{'key': None, 'value': {'customer_id': 94, 'view_time': 53, 'ts': 1787588207915}}
{'key': None, 'value': {'customer_id': 79, 'view_time': 70, 'ts': 1787588208015}}

pydbsp Z-set (as a list of pairs):
({'key': None, 'value': {'customer_id': 94, 'view_time': 53, 'ts': 1787588207915}}, 1)
({'key': None, 'value': {'customer_id': 79, 'view_time': 70, 'ts': 1787588208015}}, 1)

Output records:
{'key': None, 'value': {'customer_id': 94, 'view_time': 53, 'ts': 1787588207915}}
{'key': None, 'value': {'customer_id': 79, 'view_time': 70, 'ts': 1787588208015}}


<a id="to_debezium"></a>
#### to_debezium

Conerts the resulting Z-sets into lists of records in *Debezium* format. Each record with a positive weight `w` is converted into `w` output records with Debezium `op` set to `c`. Each record with a negative weight `w` is converted into `w` output records with Debezium `op` set to `d`.

Here is an example:


In [6]:
sink_tn = Tn.source(click_source_str).to_zSet(Tn.from_debezium)._peek().sink(sink_str)
#
tn = Tn.build(sink_tn).from_zSet(Tn.to_debezium)
#
m_list = debezium_click_generator.generate(1, w=1) + debezium_click_generator.generate(1, w=-1)

print("Input records:")
for m in m_list:
    print(m)

print("\npydbsp Z-set (as a list of pairs):")
m_list = tn.process({click_source_str: m_list})[sink_str]

print("\nOutput records:")
for m in m_list:
    print(m)


Input records:
{'key': None, 'value': {'customer_id': 99, 'view_time': 24, 'ts': 1787588207915, 'before': None, 'after': {'customer_id': 99, 'view_time': 24, 'ts': 1787588207915}, 'op': 'c'}}
{'key': None, 'value': {'customer_id': 70, 'view_time': 36, 'ts': 1787588208015, 'before': {'customer_id': 70, 'view_time': 36, 'ts': 1787588208015}, 'after': None, 'op': 'd'}}

pydbsp Z-set (as a list of pairs):
({'key': None, 'value': {'customer_id': 99, 'view_time': 24, 'ts': 1787588207915}}, 1)
({'key': None, 'value': {'customer_id': 70, 'view_time': 36, 'ts': 1787588208015}}, -1)

Output records:
{'key': None, 'value': {'customer_id': 99, 'view_time': 24, 'ts': 1787588207915, 'before': None, 'after': {'customer_id': 99, 'view_time': 24, 'ts': 1787588207915}, 'op': 'c'}}
{'key': None, 'value': {'customer_id': 70, 'view_time': 36, 'ts': 1787588208015, 'before': {'customer_id': 70, 'view_time': 36, 'ts': 1787588208015}, 'after': None, 'op': 'd'}}


<a id="_to_records"></a>
#### _to_records

Converts the resulting Z-sets into lists of pairs of a record and its weight (only makes sense for *TopologyNode*, not for *Streams*)

Here is an example:


In [7]:
sink_tn = Tn.source(click_source_str).to_zSet(Tn._from_records)._peek().sink(sink_str)
#
tn = Tn.build(sink_tn).from_zSet(Tn._to_records)
#
m_w_tuple_list = weights_click_generator.generate(1, w=1) + weights_click_generator.generate(1, w=-1)

print("Input record/weight pairs:")
for m_w_tuple in m_w_tuple_list:
    print(m_w_tuple)

print("\npydbsp Z-set (as a list of pairs):")
m_w_tuple_list = tn.process({click_source_str: m_w_tuple_list})[sink_str]

print("\nOutput record/weight pairs:")
for m_w_tuple in m_w_tuple_list:
    print(m_w_tuple)


Input record/weight pairs:
({'key': None, 'value': {'customer_id': 22, 'view_time': 33, 'ts': 1787588207915}}, 1)
({'key': None, 'value': {'customer_id': 34, 'view_time': 33, 'ts': 1787588208015}}, -1)

pydbsp Z-set (as a list of pairs):
({'key': None, 'value': {'customer_id': 22, 'view_time': 33, 'ts': 1787588207915}}, 1)
({'key': None, 'value': {'customer_id': 34, 'view_time': 33, 'ts': 1787588208015}}, -1)

Output record/weight pairs:
({'key': None, 'value': {'customer_id': 22, 'view_time': 33, 'ts': 1787588207915}}, 1)
({'key': None, 'value': {'customer_id': 34, 'view_time': 33, 'ts': 1787588208015}}, -1)


---
<a id="pack_unpack"></a>
## Pack/unpack

Kafi Streams has to overcome another difficulty to integrate with pydbsp. In pydbsp, *Z-sets* are implemented naturally as Python dictionaries. This, for instance, is a Z-set where record `A` has weight `1` and record `B` has weight `2`:
```
{"A": 1, "B": 2}
```
But what happens if, as in the case of Kafi Streams, records are dictionaries themselves? In Python, dictionaries are non-hashable, so they cannot serve as keys of a dictionary. Hence they also cannot serve as keys of dictionaries modeling Z-sets!

Kafi Streams works around this issue by providing two hooks:
* [pack_fun](#pack_fun) - pack an "unpacked"/non-hashable record (typically, a dictionary) into a "packed"/hashable record 
* [unpack_fun](#unpack_fun) - unpack a "packed"/hashable record back to the original "unpacked"/non-hashable record

We found out that `msgpack` yields the by far best performance for "packing" and "unpacking", but you are free to configure your own custom functions here (e.g. using `frozendict` etc.).


<a id="pack_fun"></a>
### pack_fun

Packs an "unpacked"/non-hashable record (typically, a dictionary) into a "packed"/hashable record.

The default `pack_fun` is:
```python
default_pack_fun = msgpack.packb
```


<a id="unpack_fun"></a>
### unpack_fun

Unpacks a "packed"/hashable record back to the original "unpacked"/non-hashable record.

The default `unpack_fun` is:
```python
default_unpack_fun = lambda x: msgpack.unpackb(x, strict_map_key=False)
```
...where `strict_map_key=False` is required to also allow integers as keys for flexibility.


---
<a id="state"></a>
## State

Since pydbsp is doing the heavy lifting of the stream processing in Kafi Streams, the state of a Kafi Streams topology is the state of the corresponding pydbsp circuit.

This section is about the methods in *TopologyNode* that deal with that state.

The *Streams* class uses these methods for ensuring [fault tolerance](checkpointing.ipynb) with checkpointing.


<a id="get_state"></a>
### get_state()

Gets the underlying pydbsp circuit state:
```python
def get_state(self):
    """The underlying evaluator (circuit state).
    
    Return:
        evaluator: the evaluator of this topology node"""
    return self._evaluator
```


<a id="set_state"></a>
### set_state()

Replaces the underlying pydbsp circuit state:
```python
def set_state(self, evaluator):
    """Replace the underlying evaluator (circuit state).
    
    Args:
        evaluator: pydbsp Evaluator running the circuit"""
    self._evaluator = evaluator
```


<a id="load_state"></a>
### load_state()

Restores the state from bytes produced by `save_state()`:
```python
def load_state(self, serialized_state_bytes):
    """Restore state from bytes produced by save_state.
    
    Args:
        serialized_state_bytes: bytes previously produced by save_state"""
    evaluator = cloudpickle.loads(serialized_state_bytes)
    #
    self.set_state(evaluator)
```


<a id="save_state"></a>
### save_state()

Serializes the current state to bytes:
```python
def save_state(self):
    """Serialize the current state to bytes.
    
    Returns:
        serialized_state_bytes: the current state of this topology node in bytes"""
```


<a id="get_state_size"></a>
### get_state_size()

Gets the size of the serialized state in bytes:
```python
def get_state_size(self):
    """Size of the serialized state in bytes.
    
    Returns:
        state_size_int: the size in bytes of the serialized state"""
```
